# Tu RAG

**Sesión 9 · IA Intermedio · EstudIA 2026**

Construye un RAG con documentos que tú elijas y mídelo con preguntas que escribas tú: qué tan bien encuentra los
pedazos que sirven y qué tan bien responde.

| Camino | Para quién | Qué haces |
|---|---|---|
| **A. Tus documentos** | Tienes documentos de tu proyecto o de tu trabajo: manuales, guías, normas, preguntas frecuentes, tus apuntes | Un RAG con ellos y de 5 a 10 preguntas tuyas con su respuesta |
| **B. Práctica** | Todavía no tienes documentos | Un RAG con la ley de protección de datos personales (Ley 1581 de 2012) |

**Sin datos personales.** Usa documentos públicos o tuyos que no traigan nombres de personas, documentos de identidad,
teléfonos ni correos. Antes de llegar al modelo, la plataforma revisa el texto y no envía lo que tenga datos personales.

Cada resultado queda en tu **registro de experimentos**. Cuando termines, **entrega el notebook** desde la última celda.

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU.

In [ ]:
# Preparación: librerías y tu clave. Ejecútala una vez (instalar tarda un momento).
!pip -q install "langchain-openai>=1.8" "langchain-qdrant>=1.2" "langchain-community>=0.4" "langchain-huggingface>=1.3" rank_bm25 "pypdf>=5"
import os, re, csv, json, time, getpass, urllib.request, urllib.error, warnings
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s09")
PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import openai

_CLAVE = os.environ.get("ESTUDIA_CLAVE") or getpass.getpass(
    "Pega tu clave (plataforma › Tu clave para los modelos de lenguaje › Copiar) y oprime Enter: ").strip()
cliente = openai.OpenAI(base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE, max_retries=3)
try:
    MODELOS = [m.id for m in cliente.models.list().data if getattr(m, "tipo", "chat") == "chat"]
    print("Listo. Modelos de lenguaje disponibles:", ", ".join(MODELOS))
except openai.AuthenticationError:
    print("La clave no es válida. Cópiala otra vez de la plataforma y vuelve a ejecutar esta celda.")
except openai.APIConnectionError:
    print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")

In [ ]:
# @title Funciones auxiliares: modelos de lenguaje, embeddings, Qdrant, buscar, medir, juez y registro
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

import torch
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
torch.set_num_threads(max(1, os.cpu_count() or 1))

def modelo_de_lenguaje(nombre, temperatura=0, max_tokens=400, **extra):
    """Un modelo de lenguaje de la plataforma, para usarlo con LangChain."""
    return ChatOpenAI(model=nombre, base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE, temperature=temperatura,
                      max_tokens=max_tokens, max_retries=6, timeout=120, **extra)

def preguntar(modelo, pregunta, sistema=None, mostrar=True, **opciones):
    """Una pregunta a un modelo, sin contexto. Devuelve la respuesta y muestra el tiempo y el costo."""
    mensajes = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": pregunta}]
    inicio = time.time()
    try:
        r = cliente.chat.completions.create(model=modelo, messages=mensajes, **opciones)
    except openai.APIStatusError as e:
        print("No se pudo:", (e.body or {}).get("message", e))
        return ""
    texto = r.choices[0].message.content or ""
    if mostrar:
        u = r.usage.model_dump()
        segundos = f"{time.time() - inicio:.1f}".replace(".", ",")
        costo = f"{u.get('costo_usd', 0):.6f}".replace(".", ",")
        print(f"{texto}\n  ({modelo} · {segundos} s · {u['prompt_tokens']} tokens de entrada y "
              f"{u['completion_tokens']} de salida · US$ {costo})")
    return texto

def cuanto_llevo():
    """Lo que llevas gastado hoy en la API de modelos de lenguaje."""
    req = urllib.request.Request(f"{PLATAFORMA}/v1/usage", headers={"Authorization": f"Bearer {_CLAVE}",
                                                                    "User-Agent": "EstudIA-notebook/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        u = json.loads(r.read())
    hoy, limite = f"{u['hoy_usd']:.4f}".replace(".", ","), f"{u['limite_diario_usd']:.2f}".replace(".", ",")
    print(f"Hoy llevas US$ {hoy} de US$ {limite} ({u['solicitudes_hoy']} solicitudes).")

_CON_PREFIJOS = []

class ConPrefijos(Embeddings):
    """Embeddings de la familia e5: “query: ” antes de cada pregunta y “passage: ” antes de cada texto (sesión 6).
    Guarda los vectores de las preguntas: preparar() los calcula todos de una vez."""
    def __init__(self, base, consulta="query: ", documento="passage: "):
        self.base, self.consulta, self.documento, self._cache = base, consulta, documento, {}
        _CON_PREFIJOS.append(self)
    def embed_documents(self, textos):
        return self.base.embed_documents([self.documento + t for t in textos])
    def embed_query(self, texto):
        if texto not in self._cache:
            self._cache[texto] = self.base.embed_query(self.consulta + texto)
        return self._cache[texto]
    def preparar(self, preguntas):
        nuevas = [p for p in dict.fromkeys(preguntas) if p not in self._cache]
        if nuevas:
            self._cache.update(zip(nuevas, self.base.embed_documents([self.consulta + p for p in nuevas])))

def embeddings_locales(nombre="intfloat/multilingual-e5-small"):
    """Un modelo de embeddings de Hugging Face que corre en Colab (sin GPU también)."""
    from langchain_huggingface import HuggingFaceEmbeddings
    return ConPrefijos(HuggingFaceEmbeddings(model_name=nombre, encode_kwargs={"normalize_embeddings": True,
                                                                              "batch_size": 32}))

def embeddings_api(nombre="intfloat/multilingual-e5-large"):
    """Un modelo de embeddings más grande, a través de la plataforma (cuesta, aunque muy poco)."""
    return ConPrefijos(OpenAIEmbeddings(model=nombre, base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE,
                                        check_embedding_ctx_length=False, chunk_size=128, max_retries=6))

def coleccion(documentos, embeddings, nombre="documentos"):
    """Una colección de Qdrant en la memoria de Colab con los vectores de los documentos."""
    inicio = time.time()
    c = QdrantVectorStore.from_documents(documentos, embeddings, location=":memory:", collection_name=nombre)
    print(f"{miles(len(documentos))} documentos en Qdrant en {time.time() - inicio:.0f} s.")
    return c

def buscar_hibrido(listas, k=5, c=60):
    """Combina varias listas de resultados (de buscadores distintos): gana lo que sale arriba en varias."""
    puntaje, docs = {}, {}
    for lista in listas:
        for puesto, d in enumerate(lista):
            clave = d.metadata.get("id", d.page_content[:80])
            puntaje[clave] = puntaje.get(clave, 0) + 1 / (c + puesto + 1)
            docs[clave] = d
    return [docs[i] for i in sorted(puntaje, key=lambda i: -puntaje[i])[:k]]

def preparar_preguntas(preguntas):
    for e in _CON_PREFIJOS:
        e.preparar(list(preguntas))

def medir_busqueda(buscar, preguntas, correctos, ks=(1, 3, 5), clave="id"):
    """En qué fracción de las preguntas el documento correcto sale entre los primeros k resultados."""
    preparar_preguntas(preguntas)
    aciertos = {k: 0 for k in ks}
    for p, ok in zip(preguntas, correctos):
        ids = [d.metadata.get(clave) for d in buscar(p)]
        ok = ok if isinstance(ok, (set, list, tuple)) else {ok}
        for k in ks:
            aciertos[k] += any(i in ok for i in ids[:k])
    return {f"en los primeros {k}": aciertos[k] / len(preguntas) for k in ks}

def puntos_busqueda(buscar, preguntas, correctos, k=5):
    """Como la plataforma: 1 punto si el correcto sale primero, 1/2 si sale segundo… 0 si no está entre los k primeros."""
    preparar_preguntas(preguntas)
    total = 0.0
    for p, ok in zip(preguntas, correctos):
        ids = [d.metadata.get("id") for d in buscar(p)][:k]
        total += 1 / (ids.index(ok) + 1) if ok in ids else 0.0
    return total / len(preguntas)

def responder_lote(cadena, buscar, preguntas, formato, k=5, simultaneas=4):
    """Busca y responde muchas preguntas a la vez. Devuelve (respuestas, documentos de cada pregunta)."""
    preparar_preguntas(preguntas)
    docs = [buscar(p)[:k] for p in preguntas]
    salidas = cadena.batch([{"contexto": formato(d), "pregunta": p} for d, p in zip(docs, preguntas)],
                           config={"max_concurrency": simultaneas}, return_exceptions=True)
    errores = [s for s in salidas if not isinstance(s, str)]
    if errores:
        print(f"{len(errores)} preguntas no se pudieron responder; por ejemplo: {str(errores[0])[:200]}")
    return [" ".join(s.split()) if isinstance(s, str) else "" for s in salidas], docs

JUEZ = ("Compara la respuesta de un sistema con la respuesta correcta a una pregunta. Escribe solo SI si dice lo "
        "mismo en lo esencial (aunque use otras palabras o agregue detalles) o NO si dice otra cosa, si le falta el "
        "dato principal o si dice que no sabe.")

def juez(preguntas, esperadas, respuestas, modelo="mistralai/mistral-small-3.2-24b-instruct"):
    """Un modelo de lenguaje como juez: ¿cada respuesta dice lo mismo que la esperada? Aproximado (y cuesta)."""
    cadena = (ChatPromptTemplate.from_messages([("system", JUEZ), ("human", "Pregunta: {p}\nRespuesta correcta: {e}\n"
                                                                          "Respuesta del sistema: {a}")])
              | modelo_de_lenguaje(modelo, max_tokens=5) | StrOutputParser())
    salidas = cadena.batch([{"p": p, "e": e, "a": a or "(sin respuesta)"} for p, e, a in zip(preguntas, esperadas, respuestas)],
                           config={"max_concurrency": 4}, return_exceptions=True)
    errores = [s for s in salidas if not isinstance(s, str)]
    if errores:
        print(f"El juez no pudo revisar {len(errores)} respuestas (cuentan como no aceptadas); por ejemplo: "
              f"{str(errores[0])[:200]}")
    return [isinstance(s, str) and s.strip().upper().startswith("S") for s in salidas]

from IPython.display import display
REGISTRO = "experimentos.csv"

def registrar(**datos):
    """Agrega una fila a tu registro de experimentos (el archivo experimentos.csv): qué probaste y qué salió."""
    fila = pd.DataFrame([{"hora": time.strftime("%H:%M"), **datos}])
    tabla = pd.concat([pd.read_csv(REGISTRO), fila], ignore_index=True) if os.path.exists(REGISTRO) else fila
    tabla.to_csv(REGISTRO, index=False)

def ver_registro():
    """Muestra el registro de experimentos, el más reciente abajo."""
    if not os.path.exists(REGISTRO):
        print("Todavía no hay experimentos en el registro.")
        return
    tabla = pd.read_csv(REGISTRO)
    display(tabla.style.format(precision=1, decimal=",", thousands=".", na_rep="").hide(axis="index"))

---
## A. Tu RAG con tus documentos

Sube tus documentos (`.pdf`, `.txt` o `.md`) a la carpeta `mis_documentos` (ícono de carpeta, a la izquierda de Colab;
crea la carpeta con clic derecho › *Nueva carpeta*). En `MIS_PREGUNTAS`, escribe de 5 a 10 preguntas que tus
documentos respondan, con la respuesta que esperas.

In [ ]:
MIS_PREGUNTAS = [
    ("Escribe aquí una pregunta que tus documentos respondan.", "Y la respuesta que esperas."),
]
MODELO = "mistralai/mistral-small-3.2-24b-instruct"   # o cualquiera de MODELOS
TAMANO = 1000           # caracteres por pedazo

def leer(ruta):
    if ruta.lower().endswith(".pdf"):
        from pypdf import PdfReader
        return "\n".join(p.extract_text() or "" for p in PdfReader(ruta).pages)
    return open(ruta, encoding="utf-8", errors="ignore").read()

rutas = sorted(os.path.join(r, f) for r, _, fs in os.walk("mis_documentos") for f in fs
               if f.lower().endswith((".pdf", ".txt", ".md")))
if rutas and not MIS_PREGUNTAS[0][0].startswith("Escribe aquí"):
    docs_a = [Document(page_content=leer(r), metadata={"fuente": os.path.basename(r)}) for r in rutas]
    pedazos_a = RecursiveCharacterTextSplitter(chunk_size=TAMANO, chunk_overlap=TAMANO // 10).split_documents(docs_a)
    print(f"{len(rutas)} documentos, {len(pedazos_a)} pedazos de máximo {miles(TAMANO)} caracteres.")
    EMBEDDINGS = embeddings_locales("intfloat/multilingual-e5-small")
    mis_docs = coleccion(pedazos_a, EMBEDDINGS, "mis_documentos")
    plantilla_a = ChatPromptTemplate.from_messages([
        ("system", "Responde usando solo el contexto, en una o dos frases. Al final, entre corchetes, di de qué "
                   "documento sacaste la respuesta. Si no está en el contexto, di que no lo encuentras."),
        ("human", "Contexto:\n{contexto}\n\nPregunta: {pregunta}")])
    cadena_a = plantilla_a | modelo_de_lenguaje(MODELO) | StrOutputParser()
    formato_a = lambda docs: "\n\n".join(f"[{d.metadata['fuente']}] {d.page_content}" for d in docs)
    preguntas_a = [p for p, _ in MIS_PREGUNTAS]
    respuestas_a, docs_a_usados = responder_lote(cadena_a, lambda p: mis_docs.similarity_search(p, k=4), preguntas_a,
                                                 formato_a)
    ok_a = juez(preguntas_a, [e for _, e in MIS_PREGUNTAS], respuestas_a)
    for (p, e), r, ds, bien in zip(MIS_PREGUNTAS, respuestas_a, docs_a_usados, ok_a):
        print(f"{'✓' if bien else '✗'} {p}\n   esperada: {e}\n   tu RAG:   {r}\n   buscó en: "
              f"{', '.join(d.metadata['fuente'] for d in ds)}\n")
    print(f"El juez acepta {sum(ok_a)} de {len(ok_a)} ({pct(sum(ok_a) / len(ok_a))}).")
    registrar(camino="A", modelo=MODELO, pedazo=TAMANO, preguntas=len(ok_a), respuestas_ok=100 * sum(ok_a) / len(ok_a))
else:
    print("Camino A: sube tus documentos a la carpeta mis_documentos y escribe tus preguntas en MIS_PREGUNTAS. "
          "Si vas por el camino B, sigue a la parte B.")

**Lo que observé** (doble clic aquí y responde): ¿encontró los pedazos que sirven? Cuando respondió mal, ¿fue culpa
del buscador (no trajo el pedazo) o del modelo (lo tenía y respondió otra cosa)? Prueba otro `TAMANO` o más pedazos
en el contexto y anótalo.

---
## B. Práctica: la ley de protección de datos personales

Los 30 artículos de la Ley 1581 de 2012, como los publica la Secretaría del Senado, y 10 preguntas con el artículo que
las responde. Te falta completar una instrucción: qué debe decir el modelo cuando la respuesta no está en la ley (si
te atascas, abre la **pista** que está debajo de la celda).

In [ ]:
ley = json.load(open(descargar("ley1581.json"), encoding="utf-8"))
docs_b = [Document(page_content=f"Artículo {a['articulo']}. {a['texto']}", metadata={"articulo": a["articulo"]})
          for a in ley]
preguntas_b = pd.read_csv(descargar("preguntas_ley1581.csv"))
EMBEDDINGS_B = embeddings_locales("intfloat/multilingual-e5-small")
ley_qdrant = coleccion(docs_b, EMBEDDINGS_B, "ley1581")

# Completa: qué debe responder el modelo cuando la respuesta no está en los artículos del contexto.
SI_NO_ESTA = ...

if SI_NO_ESTA is ...:
    print("Completa SI_NO_ESTA y vuelve a ejecutar. Si te atascas, abre la pista de abajo.")
else:
    INSTRUCCIONES_B = ("Respondes preguntas sobre la Ley 1581 de 2012 usando solo los artículos del contexto, en una o "
                       "dos frases, y citas el artículo entre corchetes. " + SI_NO_ESTA)
    cadena_b = (ChatPromptTemplate.from_messages([("system", INSTRUCCIONES_B),
                                                  ("human", "Contexto:\n{contexto}\n\nPregunta: {pregunta}")])
                | modelo_de_lenguaje("mistralai/mistral-small-3.2-24b-instruct") | StrOutputParser())
    buscar_b = lambda p: ley_qdrant.similarity_search(p, k=3)
    correctos_b = [set(int(x) for x in str(a).split(";")) for a in preguntas_b["articulos"]]
    busq_b = medir_busqueda(buscar_b, preguntas_b["pregunta"], correctos_b, ks=(1, 3), clave="articulo")
    print("Búsqueda:", {k: pct(v) for k, v in busq_b.items()})
    formato_b = lambda docs: "\n\n".join(d.page_content for d in docs)
    respuestas_b, _ = responder_lote(cadena_b, buscar_b, list(preguntas_b["pregunta"]), formato_b)
    ok_b = juez(preguntas_b["pregunta"], preguntas_b["respuesta_esperada"], respuestas_b)
    for p, r, bien in zip(preguntas_b["pregunta"], respuestas_b, ok_b):
        print(f"{'✓' if bien else '✗'} {p}\n   → {r}")
    print(f"\nEl juez acepta {sum(ok_b)} de {len(ok_b)} ({pct(sum(ok_b) / len(ok_b))}).")
    print("\nUna pregunta que la ley no responde:")
    print("→", cadena_b.invoke({"contexto": formato_b(buscar_b("¿Cuánto cuesta un pasaje de TransMilenio?")),
                                "pregunta": "¿Cuánto cuesta un pasaje de TransMilenio?"}))
    registrar(camino="B", modelo="mistralai/mistral-small-3.2-24b-instruct", busqueda_3=100 * busq_b["en los primeros 3"],
              respuestas_ok=100 * sum(ok_b) / len(ok_b))

<details><summary>Pista</summary>

Si la ley no trae la respuesta, el modelo no debe inventarla: debe decirlo. Escribe la instrucción como una frase
entre comillas, por ejemplo con esta forma:

```python
SI_NO_ESTA = "Si la respuesta no está en el contexto, di: “…”"
```
Después, mira qué responde a la pregunta sobre TransMilenio.
</details>

**Lo que observé** (doble clic aquí y responde): ¿qué preguntas falló? ¿El buscador trajo el artículo correcto? ¿Qué
respondió a la pregunta que la ley no responde?

---
## Tu registro y lo que avanzaste

In [ ]:
ver_registro()
cuanto_llevo()

**Lo que avancé** (doble clic aquí y responde en dos o tres líneas por pregunta):

- ¿Qué tan bien encontró tu buscador los pedazos que sirven? ¿Y qué tan bien respondió el modelo?
- ¿Cuándo inventó o respondió algo que no estaba en tus documentos?
- ¿Cómo encajaría un RAG en tu proyecto, o en tu trabajo? ¿Qué documentos usaría?

---
## Entrega

Ejecuta las dos celdas siguientes. La primera solo prepara la función; la segunda lee este notebook con sus
resultados y lo envía a la plataforma.

Si algo falla, descarga el notebook (*Archivo › Descargar › Descargar .ipynb*) y súbelo en la plataforma, en la
actividad “Entrega: tu RAG”.

In [ ]:
# @title Función para entregar el notebook (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = globals().get("_CLAVE") or os.environ.get("ESTUDIA_CLAVE", "")
ACTIVIDAD = "“Entrega: tu RAG”"

def _este_notebook():
    """El notebook tal como se ve en Colab, con sus resultados. Fuera de Colab, el archivo de ESTUDIA_NOTEBOOK."""
    if os.environ.get("ESTUDIA_NOTEBOOK"):
        with open(os.environ["ESTUDIA_NOTEBOOK"], encoding="utf-8") as f:
            return json.load(f)
    try:
        from google.colab import _message
        return _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    except Exception:
        return None

def entregar():
    """Envía este notebook, con lo que ejecutaste, a la plataforma del curso."""
    global _CLAVE
    cuaderno = _este_notebook()
    if not cuaderno:
        print("No pude leer el notebook desde aquí. Descárgalo (Archivo › Descargar › Descargar .ipynb) y súbelo "
              f"en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    if not _CLAVE:
        _CLAVE = getpass.getpass(f"Pega tu clave (plataforma › {ACTIVIDAD} › Copiar) y oprime Enter: ").strip()
    req = urllib.request.Request(f"{PLATAFORMA}/api/entregas/s09", data=json.dumps({"notebook": cuaderno}).encode(),
                                 method="POST", headers={"Content-Type": "application/json",
                                                         "Authorization": f"Bearer {_CLAVE}",
                                                         "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""
        print("No se pudo entregar:", res.get("error", f"error {e.code}"))
        print(f"Si no se arregla, descarga el notebook y súbelo en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    modelos = ", ".join(res["modelos"]) or "ninguno"
    print(f"Entrega recibida. Celdas de código ejecutadas: {res['ejecutadas']} de {res['celdas']}. "
          f"Modelos de Hugging Face que encontré: {modelos}.")
    for aviso in res.get("avisos", []):
        print("Ojo:", aviso)

In [ ]:
entregar()